In [1]:
import numpy as np 
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import models , layers
from tensorflow.keras.optimizers import Adam , RMSprop
import tensorflow as tf

## read and clean the data

In [2]:
def cleanData(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

In [3]:
dataset = pd.read_csv('sentiment140.csv')
dataset = dataset[['text' , 'sentiment']]

dataset['text'] = dataset['text'].apply(cleanData)
labelE=LabelEncoder()
dataset['sentiment'] = labelE.fit_transform(dataset['sentiment'])

## tokenize and padding

In [4]:
max_word = 20000
max_len = 100
tokenizer = Tokenizer(num_words=max_word)
tokenizer.fit_on_texts(dataset['text'])

sequences = tokenizer.texts_to_sequences(dataset['text'])
X = pad_sequences(sequences, maxlen=max_len)
y = dataset['sentiment']

x_train , x_test , y_train , y_test = train_test_split(X , y , test_size=0.2, random_state=42)
print(x_train , y_train)

[[   0    0    0 ...    4  157  159]
 [   0    0    0 ...  697   21  112]
 [   0    0    0 ...   14    3  130]
 ...
 [   0    0    0 ...  242   54  291]
 [   0    0    0 ...  209 3464   86]
 [   0    0    0 ... 1927   14  341]] 14307    0
17812    0
11020    0
15158    0
24990    1
        ..
6265     0
11284    1
38158    1
860      0
15795    1
Name: sentiment, Length: 32000, dtype: int64


## load GloVe embeddings

In [5]:
embedding_dimensions = 100
embedding_index = {}

with open('glove.6B.100d.txt' , encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.array(values[1:] , dtype='float32')
        embedding_index[word] = vector

word_index = tokenizer.word_index

embedding_matrix = np.zeros((max_word, embedding_dimensions))

for word , i in word_index.items():
    if i < max_word:
        vector = embedding_index.get(word)
        if vector is not None:
            embedding_matrix[i] = vector

In [6]:
def build_CNN(optimizer = 'adam' , drop_out = 0):
    model = models.Sequential()
    model.add(
        layers.Embedding(
            input_dim=max_word,
            output_dim=embedding_dimensions,
            weights=[embedding_matrix],
            trainable=False
        )
    )
    model.add(layers.Conv1D(128, kernel_size=5, activation='relu'))
    model.add(layers.MaxPooling1D(2))

    model.add(layers.Conv1D(64, kernel_size=5, activation='relu'))
    model.add(layers.GlobalAveragePooling1D())

    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(drop_out))

    model.add(layers.Dense(1, activation='sigmoid'))
    if optimizer == 'adam':
        optimizer = Adam()
    elif optimizer == 'rmsprop':
        optimizer = RMSprop()

    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
Models_to_train = []
results = {}

x_train , x_val , y_train , y_val = train_test_split(x_train , y_train , test_size=0.1, random_state=42)

tf.config.run_functions_eagerly(True)

model1 = build_CNN(optimizer= 'adam', drop_out=0.2)
model2 = build_CNN(optimizer= 'adam', drop_out=0.5)
model3 = build_CNN(optimizer= 'adam', drop_out=0.7)
model4 = build_CNN(optimizer= 'rmsprop', drop_out=0.2)
model5 = build_CNN(optimizer= 'rmsprop', drop_out=0.5)
Models_to_train.append(model1)
Models_to_train.append(model2)
Models_to_train.append(model3)
Models_to_train.append(model4)
Models_to_train.append(model5)
# print(results)

In [8]:
for i in range(len(Models_to_train)):
    print(f"-----------------------Model {i+1}training-------------------------")
    Models_to_train[i].fit(x_train, y_train, validation_data=(x_val, y_val), epochs=5, batch_size=32, verbose=1)
    loss , acc = Models_to_train[i].evaluate(x_test, y_test)
    results[Models_to_train[i]] = (loss, acc)

print(results)

-----------------------Model 1training-------------------------
Epoch 1/5


c:\Users\Ahmed Mohamed\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\python\data\ops\structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


900/900 ━━━━━━━━━━━━━━━━━━━━ 45s 50ms/step - accuracy: 0.6668 - loss: 0.6057 - val_accuracy: 0.7184 - val_loss: 0.5473
Epoch 2/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 44s 49ms/step - accuracy: 0.7439 - loss: 0.5178 - val_accuracy: 0.7250 - val_loss: 0.5382
Epoch 3/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 44s 49ms/step - accuracy: 0.7758 - loss: 0.4733 - val_accuracy: 0.7597 - val_loss: 0.4996
Epoch 4/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 45s 49ms/step - accuracy: 0.8012 - loss: 0.4287 - val_accuracy: 0.7481 - val_loss: 0.5144
Epoch 5/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 45s 50ms/step - accuracy: 0.8310 - loss: 0.3811 - val_accuracy: 0.7422 - val_loss: 0.5411
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.7424 - loss: 0.5408
-----------------------Model 2training-------------------------
Epoch 1/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 47s 53ms/step - accuracy: 0.6634 - loss: 0.6103 - val_accuracy: 0.7300 - val_loss: 0.5416
Epoch 2/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 51s 57ms/step - accuracy: 0.7398 - loss: 0.5252 - val_acc

In [12]:
for i in range(len(Models_to_train)):
    print(f'{Models_to_train[i].name}: Loss = {results[Models_to_train[i]][0]}, Accuracy = {results[Models_to_train[i]][1]}')

sequential: Loss = 0.5408290028572083, Accuracy = 0.7423750162124634
sequential_1: Loss = 0.5603933930397034, Accuracy = 0.7441250085830688
sequential_2: Loss = 0.5577160716056824, Accuracy = 0.7319999933242798
sequential_3: Loss = 0.5103774666786194, Accuracy = 0.7577499747276306
sequential_4: Loss = 0.5186201333999634, Accuracy = 0.7488750219345093


## the most efficient model is model number 4 (optimizer = RMSprop , drop out = 0.2) with accuracy = 0.76 and loss = 0.51